# MLB's drafting proposal: what the 1 seed is worth, and how to draft

Code behind the article on MLB's 2026 CBA proposal to let the top seeds draft their playoff opponents (from 2027), plus the 154-game season and best-of-7 Division Series (from 2029).

**How it works:** every team gets a *talent score* (its schedule-adjusted record from that season, pulled toward .500 to remove luck; see `src/ratings.py` and `src/sim.py`). Each of 13 seasons (2013–2026, excl. 2020) is replayed 20,000 times on the 2026 schedule, and every simulated regular season is run through each postseason format below (`src/bracket.py`).
For each seed (1-6): how often it reaches the Division Series, the LCS, wins the pennant and wins the World Series, under:

| Format | Season | DS length | Drafting |
|---|---|---|---|
| current | 162 | best-of-5 | none |
| 2027 | 162 | best-of-5 | #3 picks its Wild Card opponent, #1 picks its DS opponent |
| 2029 | 154 | best-of-7 | same drafting |

Drafting rules: picks by talent score (perfect scouting), by record, by head-to-head, and record + head-to-head.

The 1 seed is usually a better team than the 2 seed, so the raw gap between them is partly talent. The bracket's own effect is **how much the 1-vs-2 gap grows** when drafting is added: same simulated teams, only the bracket differs.

20,000 simulated seasons per talent season (260,000 total), paired within each season length.

In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path

sys.path.insert(0, '..')
from src.sim import simulate_games, shorten_schedule
from src.bracket import run_postseason, FORMATS, format_12_grid
from src.evaluate import season_setup

RAW = Path('../data/raw')
RESULTS = Path('../results')
RESULTS.mkdir(exist_ok=True)
games = pd.read_csv(RAW / 'games_1998_2026.csv')
standings = pd.read_csv(RAW / 'standings_1998_2026.csv')
rng = np.random.default_rng(2014)
N, BATCH = 20000, 5000
ALL_SEASONS = [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2021, 2022, 2023, 2024, 2025, 2026]
g154 = shorten_schedule(games[games['season'] == 2026])
RULES = {'talent': {'talent': 1}, 'record': {'record': 1}, 'head-to-head': {'h2h': 1},
         'record + head-to-head': {'record': 1, 'h2h': 1}}
RUNS = {'current': (162, dict())}
for r, wts in RULES.items():
    RUNS[f'2027, draft by {r}'] = (162, dict(ds_rule='pick', pick_weights=wts))
for r in ('talent', 'record'):
    RUNS[f'2029, draft by {r}'] = (154, dict(ds_length=7, ds_rule='pick', pick_weights=RULES[r]))
for name, (_, opts) in RUNS.items():
    FORMATS[name] = (lambda o: lambda lg, c: format_12_grid(lg, c, **o))(opts)
list(RUNS)

['current',
 '2027, draft by talent',
 '2027, draft by record',
 '2027, draft by head-to-head',
 '2027, draft by record + head-to-head',
 '2029, draft by talent',
 '2029, draft by record']

In [2]:
rows = []
for season in ALL_SEASONS:
    for length, sched_games in ((162, None), (154, g154)):
        T = season_setup(games, standings, season, schedule_season=2026, schedule_games=sched_games)
        st = np.asarray(T['strength'])
        for _ in range(N // BATCH):
            w, l, h2h = simulate_games(T['sched'], T['strength'], T['home_edge'], BATCH, rng, head_to_head=True)
            for name, (ln, _) in RUNS.items():
                if ln != length:
                    continue
                res = run_postseason(name, T['leagues'], w, l, h2h, T['strength'], T['home_edge'], rng)
                base = {'talent from': season, 'format': name}
                for k in range(6):
                    per = {'reaches Division Series': [], 'reaches LCS': [], 'wins pennant': [], 'wins World Series': []}
                    for r, pen in zip(res['leagues'], res['pennants']):
                        team = r['seeds'][:, k]
                        per['reaches Division Series'].append((r['reached_ds'] == team[:, None]).any(axis=1).mean())
                        per['reaches LCS'].append((r['reached_lcs'] == team[:, None]).any(axis=1).mean())
                        per['wins pennant'].append((pen == team).mean())
                        per['wins World Series'].append((res['champion'] == team).mean())
                    for what, vals in per.items():
                        rows.append({**base, 'seed': k + 1, 'measure': what, 'value': np.mean(vals)})
                if 'picks' in res['leagues'][0]:
                    pk = np.concatenate([r['picks'] for r in res['leagues']])
                    rows.append({**base, 'seed': 3, 'measure': 'picks the less talented opponent', 'value': (st[pk[:, 0]] < st[pk[:, 1]]).mean()})
                    rows.append({**base, 'seed': 1, 'measure': 'picks the less talented opponent', 'value': (st[pk[:, 2]] < st[pk[:, 3]]).mean()})
seeds = pd.DataFrame(rows).groupby(['measure', 'seed', 'format'])['value'].mean().unstack()[list(RUNS)] * 100
seeds.round(2)

format                                 current  2027, draft by talent  \
measure                          seed                                   
picks the less talented opponent 1         NaN                 100.00   
                                 3         NaN                 100.00   
reaches Division Series          1      100.00                 100.00   
                                 2      100.00                 100.00   
                                 3       57.11                  59.21   
                                 4       57.52                  55.50   
                                 5       42.48                  43.18   
                                 6       42.89                  42.11   
reaches LCS                      1       57.27                  62.69   
                                 2       55.81                  50.22   
                                 3       25.85                  25.09   
                                 4       25.28                  26.43   
                                 5       17.45                  18.78   
                                 6       18.34                  16.79   
wins World Series                1       18.10                  18.96   
                                 2       13.28                  12.47   
                                 3        5.23                   5.26   
                                 4        6.41                   6.35   
                                 5        3.76                   3.91   
                                 6        3.22                   3.05   
wins pennant                     1       33.19                  34.73   
                                 2       26.59                  25.07   
                                 3       11.37                  11.45   
                                 4       13.03                  12.99   
                                 5        8.32                   8.62   
                                 6        7.50                   7.14   

format                                 2027, draft by record  \
measure                          seed                          
picks the less talented opponent 1                     66.57   
                                 3                     61.26   
reaches Division Series          1                    100.00   
                                 2                    100.00   
                                 3                     57.16   
                                 4                     57.59   
                                 5                     42.55   
                                 6                     42.70   
reaches LCS                      1                     60.78   
                                 2                     52.27   
                                 3                     24.22   
                                 4                     27.62   
                                 5                     18.51   
                                 6                     16.60   
wins World Series                1                     18.80   
                                 2                     12.68   
                                 3                      5.10   
                                 4                      6.51   
                                 5                      3.83   
                                 6                      3.08   
wins pennant                     1                     34.36   
                                 2                     25.54   
                                 3                     11.14   
                                 4                     13.34   
                                 5                      8.46   
                                 6                      7.17   

format                                 2027, draft by head-to-head  \
measure                          seed                                
picks the less talente

In [3]:
# The 1 seed's edge over the 2 seed, and how much drafting adds to it (points)
gap = (seeds.xs(1, level='seed') - seeds.xs(2, level='seed')).drop('picks the less talented opponent', errors='ignore')
gap_change = gap.sub(gap['current'], axis=0)
print('1 seed minus 2 seed (points)'); display(gap.round(2))
print('change in that gap vs current (points)'); display(gap_change.round(2))

1 seed minus 2 seed (points)


format,current,"2027, draft by talent","2027, draft by record","2027, draft by head-to-head","2027, draft by record + head-to-head","2029, draft by talent","2029, draft by record"
measure,,,,,,,
reaches Division Series,0.00,0.00,0.00,0.00,0.00,0.00,0.00
reaches LCS,1.46,12.47,8.50,5.17,6.87,14.44,9.41
wins World Series,4.81,6.49,6.11,5.35,5.63,6.78,5.97
wins pennant,6.60,9.66,8.83,7.70,8.04,10.47,9.06


change in that gap vs current (points)


format,current,"2027, draft by talent","2027, draft by record","2027, draft by head-to-head","2027, draft by record + head-to-head","2029, draft by talent","2029, draft by record"
measure,,,,,,,
reaches Division Series,0.0,0.00,0.00,0.00,0.00,0.00,0.00
reaches LCS,0.0,11.01,7.05,3.71,5.41,12.98,7.95
wins World Series,0.0,1.67,1.30,0.54,0.81,1.96,1.15
wins pennant,0.0,3.06,2.23,1.09,1.44,3.87,2.46


In [4]:
seeds.reset_index().to_csv(RESULTS / 'seed_value_and_draft_rules.csv', index=False)